In [17]:
from torch.optim import optimizer


In [18]:
%unload_ext pycodestyle_magic


The pycodestyle_magic extension is not loaded.


In [19]:
import torch
from torch import nn
import torchvision
from torchvision import transforms

In [20]:
class FashionMNIST():
    def __init__(self, batch_size=64, resize=(28, 28)):
        self.batch_size = batch_size
        self.resize = resize
        self.transform = transforms.Compose([transforms.Resize(self.resize),
                                             transforms.ToTensor()])
        self.train = torchvision.datasets.FashionMNIST(root='./root', train=True, download=True,
                                                       transform=self.transform)
        self.validation = torchvision.datasets.FashionMNIST(root='./root', train=False, download=True,
                                                            transform=self.transform)

    def get_dataloader(self, train):
        data = self.train if train else self.validation
        dataloader = torch.utils.data.DataLoader(data, batch_size=self.batch_size, shuffle=train)
        return dataloader

    def training_data(self):
        return self.get_dataloader(True)

    def validation_data(self):
        return self.get_dataloader(False)

In [21]:
class Softmax_calssifier(nn.Module):
    def __init__(self, num_outputs, lr=0.03, weight_decay:float=3.0):
        super().__init__()
        self.num_outputs = num_outputs
        self.lr = lr
        self.net = nn.Sequential(nn.Flatten(), nn.LazyLinear(num_outputs))
        self.optimizer = torch.optim.SGD(self.parameters(), lr=self.lr, weight_decay=weight_decay)

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        return self.net(X)

    def loss(self, y_hat: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        fn = nn.CrossEntropyLoss()
        return fn(y_hat, y)


In [22]:
def Train_batches(dataloader, model, optimizer):

    model.train()
    for (X, y) in dataloader:
        pred = model(X)
        loss = model.loss(pred, y)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()


def validate_batches(dataloader, model):

    total_loss, correct = 0, 0
    model.eval()
    for (X, y) in dataloader:
        with torch.no_grad():
            y_hat = model(X)
            total_loss += model.loss(y_hat, y).item()
            correct += (y_hat.argmax(dim=1) == y).float().sum().item()

    total_loss /= len(dataloader)
    correct /= len(dataloader.dataset)
    print(f"validation loss: {total_loss :> 8f} and correct: {(100 * correct) :>0.1f}%")

In [23]:
data = FashionMNIST(batch_size=256)
data_train = data.training_data()
data_validation = data.validation_data()
model = Softmax_calssifier(num_outputs=10, lr=1e-1, weight_decay=5e-4)
optim = model.optimizer

for i in range(10):
    Train_batches(data_train, model, optim)
    validate_batches(data_validation, model)

validation loss:  0.654358 and correct: 77.0%
validation loss:  0.565783 and correct: 80.7%
validation loss:  0.566015 and correct: 80.0%
validation loss:  0.515970 and correct: 82.6%
validation loss:  0.511668 and correct: 82.1%
validation loss:  0.516453 and correct: 81.6%
validation loss:  0.506529 and correct: 82.3%
validation loss:  0.488648 and correct: 83.3%
validation loss:  0.511611 and correct: 82.2%
validation loss:  0.491593 and correct: 82.5%


In [24]:
validate_batches(data_validation, model)

validation loss:  0.491593 and correct: 82.5%


lr=1e-3
validation loss: 1.9156358122825623 and correct: 0.4962
validation loss: 1.6589419007301331 and correct: 0.609
validation loss: 1.4857589811086656 and correct: 0.643
validation loss: 1.363202166557312 and correct: 0.6505
validation loss: 1.2723620802164077 and correct: 0.6559
validation loss: 1.2024999260902405 and correct: 0.6601
validation loss: 1.1470255494117736 and correct: 0.6643
validation loss: 1.1018389284610748 and correct: 0.6682
validation loss: 1.064208945631981 and correct: 0.6728
validation loss: 1.0322996854782105 and correct: 0.6779
lr=1e-1
validation loss:  0.642062% and correct: 77.9%
validation loss:  0.572802% and correct: 80.4%
validation loss:  0.537184% and correct: 81.4%
validation loss:  0.530188% and correct: 81.3%
validation loss:  0.508828% and correct: 82.2%
validation loss:  0.497995% and correct: 82.6%
validation loss:  0.497446% and correct: 82.7%
validation loss:  0.487209% and correct: 82.9%
validation loss:  0.495450% and correct: 82.8%
validation loss:  0.478777% and correct: 83.4%
lr = 1e-1, weight_decay = 5e-4
